# Notebook 03: RAG Pipeline

Notebook ini mendemonstrasikan proses loading dokumen dari knowledge base, chunking, embedding generation menggunakan `multilingual-e5-small`, dan semantic search dengan FAISS.

In [9]:
import sys
import os
# Menambahkan path supaya bisa import src
sys.path.append(os.path.abspath('..'))

from src.rag_service import RAGService
import pandas as pd
from openai import OpenAI
import os
from dotenv import load_dotenv

## 1. Load Document & Inisialisasi RAG Service

In [2]:
rag = RAGService(index_dir='../faiss_index')
docs = rag.load_knowledge_base(kb_dir='../knowledge_base')
print(f'Total dokumen termuat: {len(docs)}')

if docs:
    print('\nContoh Metadata Dokumen 1:')
    print(docs[0].metadata)
    print('\nContoh Konten Dokumen 1:')
    print(docs[0].page_content[:200] + '...')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Total dokumen termuat: 12

Contoh Metadata Dokumen 1:
{'document_id': 'A002', 'type': 'addon', 'category': 'beverage', 'price': 5000, 'active': True, 'source': 'beverages.md'}

Contoh Konten Dokumen 1:
# Pilihan Minuman Tambahan

Beberapa paket kami sudah termasuk Air Mineral (seperti Paket Corporate A dan B). Namun, jika Anda membutuhkan tambahan atau upgrade minuman, berikut pilihannya:

**Air Min...


## 2. Chunking

In [3]:
chunked_docs = rag.chunk_documents(docs)
print(f'Total chunks setelah proses chunking: {len(chunked_docs)}')

if chunked_docs:
    print('\nContoh Chunk 1:')
    print(chunked_docs[0].page_content)

Total chunks setelah proses chunking: 18

Contoh Chunk 1:
# Pilihan Minuman Tambahan

Beberapa paket kami sudah termasuk Air Mineral (seperti Paket Corporate A dan B). Namun, jika Anda membutuhkan tambahan atau upgrade minuman, berikut pilihannya:

**Air Mineral Botol (330ml)**
- Tambahan Rp 3.000 / box (jika upgrade dari air gelas)
- Beli terpisah Rp 4.000 / botol

**Teh Kotak / Jus Kotak (200ml)**
- Beli terpisah Rp 5.000 / kotak

**Kopi / Teh Termos (Khusus Event Prasmanan/Meeting)**
- Rp 150.000 / termos (Kapasitas ~30 cup, sudah termasuk gula dan gelas kertas)


## 3. Generate Embedding & Build FAISS Index

In [4]:
# Ini akan mendownload model jika belum ada (sekitar 470MB)
# Proses embedding mungkin memakan waktu beberapa detik
rag.build_index(chunked_docs)
rag.save_index()

Menghasilkan embedding untuk 18 chunks...
Berhasil membuat index FAISS dengan 18 vektor.
Index berhasil disimpan di ../faiss_index/


## 4. Semantic Search (Retrieval)

In [5]:
queries = [
    'Berapa harga paket hemat yang paling murah?',
    'Apakah bisa request custom menu untuk alergi seafood?',
    'Berapa lama maksimal pemesanan sebelum hari H?'
]

for query in queries:
    results = rag.search(query, top_k=2)
    print(f'Query: "{query}"')
    for i, res in enumerate(results, 1):
        print(f'--- Hasil {i} (Source: {res.metadata.get("source", "")}, Score: {res.metadata.get("relevance_score", 0)}) ---')
        print(res.page_content)
    print('\n' + '='*50 + '\n')


Query: "Berapa harga paket hemat yang paling murah?"
--- Hasil 1 (Source: paket_hemat_a.md, Score: 0.8776) ---
# Paket Hemat A

**Harga:** Rp 18.000 / box
**Minimum Order:** 20 box
**Cocok untuk:** Arisan, Pengajian, Acara Keluarga sederhana.

## Menu Utama
- Nasi Putih
- Ayam Goreng
- Sayur Sop / Orak Arik Buncis
- Sambal Terasi
- Kerupuk Udang

## Deskripsi
Paket Hemat A adalah pilihan paling hemat untuk acara kumpul-kumpul sederhana. Dengan menu ayam goreng klasik yang disukai semua orang, paket ini memberikan solusi praktis dan lezat dengan budget yang sangat terjangkau.
--- Hasil 2 (Source: paket_hemat_b.md, Score: 0.8687) ---
# Paket Hemat B

**Harga:** Rp 23.000 / box
**Minimum Order:** 20 box
**Cocok untuk:** Arisan, Pengajian, Acara Keluarga, Ulang Tahun.

## Menu Utama
- Nasi Putih
- Ayam Bakar Kecap
- Sayur Lodeh / Capcay
- Telur Balado Separuh
- Sambal Bajak
- Kerupuk Udang
- Buah (Jeruk / Pisang)

## Deskripsi
Paket Hemat B menawarkan hidangan yang lebih lengkap dengan tam

## 5. Metadata Filtering

In [6]:
query = 'Paket untuk acara arisan'
# Filter hanya tipe produk dan category 'hemat'
filters = {'type': 'product', 'category': 'hemat'}

results_filtered = rag.search(query, top_k=1, metadata_filters=filters)

print(f'Query: "{query}"')
print(f'Filters: {filters}\n')
for i, res in enumerate(results_filtered, 1):
    print(f'--- Hasil {i} (Source: {res.metadata.get("source", "")}) ---')
    print(res.metadata)
    print()

Query: "Paket untuk acara arisan"
Filters: {'type': 'product', 'category': 'hemat'}

--- Hasil 1 (Source: paket_hemat_a.md) ---
{'document_id': 'P001', 'type': 'product', 'category': 'hemat', 'price': 18000, 'minimum_order': 20, 'event_types': ['arisan', 'pengajian', 'acara_keluarga'], 'active': True, 'source': 'paket_hemat_a.md', 'chunk_id': 0, 'relevance_score': 0.8565}



## 6. Context Construction untuk LLM

Bagian ini menunjukkan bagaimana output dari search dibentuk menjadi satu teks panjang untuk diberikan kepada LLM sebagai *context* (RAG).

In [7]:
context = rag.construct_context(results_filtered)
print('=== CONTEXT STRING ===\n')
print(context)
print('\n========================')

=== CONTEXT STRING ===

[Document 1 | Source: paket_hemat_a.md]
# Paket Hemat A

**Harga:** Rp 18.000 / box
**Minimum Order:** 20 box
**Cocok untuk:** Arisan, Pengajian, Acara Keluarga sederhana.

## Menu Utama
- Nasi Putih
- Ayam Goreng
- Sayur Sop / Orak Arik Buncis
- Sambal Terasi
- Kerupuk Udang

## Deskripsi
Paket Hemat A adalah pilihan paling hemat untuk acara kumpul-kumpul sederhana. Dengan menu ayam goreng klasik yang disukai semua orang, paket ini memberikan solusi praktis dan lezat dengan budget yang sangat terjangkau.




## 7. RAG + Groq Integration Test (Section 3.8)

Mari kita integrasikan context yang didapat dengan Groq.

In [10]:
load_dotenv()

api_key = os.getenv('GROK_API_KEY')
if api_key:
    client = OpenAI(api_key=api_key, base_url='https://api.groq.com/openai/v1')
    
    # Test pertanyaan yang butuh RAG
    pertanyaan = "Berapa minimum order untuk paket hemat A dan apa cocok buat arisan?"
    
    # Dapatkan Context dari RAG
    rag_results = rag.search(pertanyaan, top_k=2)
    rag_context = rag.construct_context(rag_results)
    
    system_prompt = f"""Anda adalah asisten virtual Nasi Kotak.\n
Jawab pertanyaan HANYA berdasarkan konteks berikut. Jika tidak ada di konteks, bilang 'Saya tidak tahu'.\n
\n
KONTEKS:\n
{rag_context}\n
"""
    
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": pertanyaan}
        ]
    )
    print("Pertanyaan:", pertanyaan)
    print("\n--- Jawaban Groq (Dengan RAG) ---")
    print(response.choices[0].message.content)
else:
    print("GROK_API_KEY belum diset di .env")

Pertanyaan: Berapa minimum order untuk paket hemat A dan apa cocok buat arisan?

--- Jawaban Groq (Dengan RAG) ---
Minimum order untuk Paket Hemat A adalah 20 box, dan Paket Hemat A cocok untuk acara arisan.


## 8. Perbandingan RAG vs Tanpa RAG (Hallucination Test) (Section 3.9)

In [11]:
if api_key:
    pertanyaan_halu = "Apa saja menu di paket corporate B dan berapa harganya?"
    
    print("\n--- 1. Tanpa RAG (Bisa Halusinasi) ---")
    response_no_rag = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {"role": "system", "content": "Kamu adalah asisten virtual Nasi Kotak."},
            {"role": "user", "content": pertanyaan_halu}
        ]
    )
    print(response_no_rag.choices[0].message.content)
    
    print("\n--- 2. Dengan RAG (Akurat) ---")
    rag_results_halu = rag.search(pertanyaan_halu, top_k=2)
    rag_context_halu = rag.construct_context(rag_results_halu)
    system_prompt_halu = f"Jawab HANYA berdasarkan konteks:\n{rag_context_halu}"
    response_with_rag = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {"role": "system", "content": system_prompt_halu},
            {"role": "user", "content": pertanyaan_halu}
        ]
    )
    print(response_with_rag.choices[0].message.content)



--- 1. Tanpa RAG (Bisa Halusinasi) ---
Menu di paket corporate B dari Nasi Kotak dapat berubah-ubah tergantung daerah dan lokasi. Namun, umumnya, paket corporate B memiliki beberapa pilihan menu yang berbeda.

Beberapa menu di paket corporate B Nasi Kotak antara lain:

- Nasi Putih, Ayam Bakar, Kacang Merah, Keju dan Sambal Kacang : Rp 45.500
- Nasi Kuning, Ayam Bakar, Telur Balado, Sosis Sapi dan Keju : Rp 45.500
- Nasi Putih, Sosis Sapi, Udang Masak Sambal Kecap dan Keju: Rp 45.500 
- Nasi Kuning, Daging Giling, Sambal Goreng, Kacang Merah dan Sambal Kecap: Rp 44.500 
- Nasi Putih, Sayur Asem, Ayam Goreng dan Telur Rebus dengan Keju dan Kacang Goreng : Rp 40.000 

Perlu diketahui bahwa harga dan menu dapat berbeda-beda tergantung pada wilayah dan lokasi. Jadi, untuk mendapatkan informasi yang akurat, silakan mencari informasi di website resmi atau kontak langsung dengan Nasi Kotak terdekat.

--- 2. Dengan RAG (Akurat) ---
Berikut adalah menu di Paket Corporate B beserta harganya:

1